# Build an AI Agent with LangChain

### What You'll Learn

This notebook builds upon the RAG concepts from the previous tutorial and introduces **AI Agents** - systems that can reason, plan, and use tools autonomously.

**Progressive Learning Path:**

1. **Review: From RAG to Agents** - Understand why we need agents
2. **Environment Setup** - Install LangChain and LangGraph
3. **Basic LLM with Tools** - Enable tool-calling capabilities
4. **Define Tools** - Create tools for agents to use (search, calculations, etc.)
5. **Build a Simple Agent** - Use LangGraph's ReAct pattern
6. **Agent Execution** - Run stateless queries
7. **Add Memory** - Make agents conversational with checkpointing
8. **Advanced Patterns** - Streaming and monitoring

### What is an AI Agent?

**Agents** are LLM-powered systems that can:
- ✅ **Reason** about which actions to take
- ✅ **Use tools** to gather information or perform actions
- ✅ **Make decisions** based on results
- ✅ **Execute multi-step plans** autonomously

Think of it like upgrading from a Q&A chatbot to an **autonomous assistant** that can actually do things!

### The Agent Architecture

**Key Differences from RAG:**

| RAG Application | Agent Application |
|----------------|------------------|
| Retrieves documents | Can use multiple tools |
| Single-step: retrieve → generate | Multi-step: reason → act → observe → repeat |
| Passive knowledge lookup | Active problem solving |
| Predictable flow | Dynamic decision making |

---

Let's build your first agent! 🤖

## Step 0: Environment Setup

Before we begin building agents, let's install all the required packages.

**Key Packages:**
- **langgraph** - Framework for building stateful, multi-agent applications
- **langchain** - Core LangChain framework
- **langchain-tavily** - Integration with Tavily search engine (our main tool)
- **langgraph-checkpoint-sqlite** - For agent memory/persistence

Run the cell below to install dependencies:

In [1]:
import getpass
import os

TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

**Key Observation:** 
- `content` may be empty or have explanatory text
- `tool_calls` contains tool invocation details:
  - `name`: Which tool to call (`tavily_search`)
  - `args`: Arguments to pass (`{"query": "weather San Francisco"}`)
  - `id`: Unique ID for tracking

**Important:** The LLM is **NOT executing the tool** - it's just telling us what tool to call! This is where agents come in.

**Observation:** The agent doesn't know your name or location because this is a **new conversation** with a different `thread_id`.

Each thread maintains its own isolated conversation history!

## Resources

**Key Packages:**
- `langgraph` - Agent orchestration framework
- `langchain` - Core LLM framework
- `langchain-tavily` - Tavily search integration

**LangSmith:**
- [LangSmith Platform](https://smith.langchain.com/) - For tracing and debugging

**Next Notebooks:**
- `6.AgenticRAG.ipynb` - Combining agents with RAG
- `7.AgentSupervisor.ipynb` - Multi-agent orchestration

---

**Congratulations!** 🎉 You've completed the Agent tutorial. You now understand:
- ✅ How agents differ from simple LLM calls
- ✅ The ReAct pattern for reasoning and acting
- ✅ Tool-calling and tool execution
- ✅ Building agents with LangGraph
- ✅ Adding memory for conversations
- ✅ Streaming agent responses

Ready to move on to **Agentic RAG**? 🚀

## Next Steps

Now that you understand agents, here are some things to explore:

### 1. **Create Custom Tools**

You can define your own tools using Python functions:

```python
from langchain.tools import tool

@tool
def calculate_square(number: int) -> int:
    """Calculate the square of a number."""
    return number ** 2

tools = [search, calculate_square]
```

### 2. **Add More Tools**

Expand your agent's capabilities:
- **Web scraping** - Extract content from websites
- **File operations** - Read/write files
- **Database queries** - Query SQL databases
- **API calls** - Weather, stocks, news, etc.
- **Python REPL** - Execute Python code

### 3. **Persistent Memory**

For production, use database-backed checkpointing:

```python
from langgraph.checkpoint.sqlite import SqliteSaver

memory = SqliteSaver.from_conn_string("checkpoints.db")
agent = create_react_agent(model, tools, checkpointer=memory)
```

### 4. **Agentic RAG** (Next Notebook!)

Combine agents with RAG:
- Agent decides when to search documents
- Agent decides when to use other tools
- More flexible than pure RAG

We'll build this in the **next notebook**: `6.AgenticRAG.ipynb`

### 5. **Multi-Agent Systems** (Advanced)

Create systems with multiple specialized agents:
- Supervisor agent coordinates sub-agents
- Each agent has specific expertise
- Agents collaborate on complex tasks

This is covered in: `7.AgentSupervisor.ipynb`

## Summary: What We Learned

Congratulations! You've built your first AI agent. Let's recap:

### Key Concepts

1. **Tools** - Functions that agents can call to perform actions
   - Search engines (Tavily)
   - APIs, databases, calculators, etc.
   - Defined with clear names and descriptions

2. **ReAct Pattern** - How agents think and act
   - **Reason** - Decide what to do
   - **Act** - Use a tool
   - **Observe** - See the result
   - **Repeat** - Until task is complete

3. **Tool-Calling** - LLMs can suggest tool usage
   - Use `.bind_tools()` to enable
   - LLM returns `tool_calls` with function name and arguments
   - Agent executor actually calls the tools

4. **LangGraph's `create_react_agent`** - High-level agent creation
   - Automatically handles ReAct loop
   - Supports multiple tools
   - Built-in streaming capabilities

5. **Memory/Checkpointing** - Conversational state
   - Use `MemorySaver` for in-memory persistence
   - `thread_id` separates conversations
   - Agents remember context across turns

### The Agent Workflow

```
User Question
     ↓
Agent Reasoning (LLM)
     ↓
Tool Selection (if needed)
     ↓
Tool Execution
     ↓
Result Observation
     ↓
Final Answer (or repeat loop)
```

### Comparison: RAG vs Agents

| Feature | RAG | Agents |
|---------|-----|--------|
| **Purpose** | Answer questions from documents | Autonomous task execution |
| **Flow** | Retrieve → Generate | Reason → Act → Observe (loop) |
| **Tools** | Vector database only | Multiple tools |
| **Steps** | Single-step (retrieval + generation) | Multi-step (dynamic) |
| **Decision Making** | Deterministic retrieval | LLM decides next action |

In [ ]:
# Start a conversation
config = {"configurable": {"thread_id": "my_conversation"}}

# Helper function for clean interaction
def chat(message: str, thread_id: str = "my_conversation"):
    """Send a message to the agent and print the response."""
    config = {"configurable": {"thread_id": thread_id}}
    input_msg = {"role": "user", "content": message}
    
    print(f"\n{'='*60}")
    print(f"YOU: {message}")
    print(f"{'='*60}\n")
    
    for step in agent.stream({"messages": [input_msg]}, config, stream_mode="values"):
        last_msg = step["messages"][-1]
        if last_msg.type == "ai":
            print(f"AGENT: {last_msg.content}\n")

# Try it out!
chat("Hi, I'm Alice and I'm interested in learning about AI agents.")
chat("Can you search for recent news about LangChain?")
chat("What was my name again?")

### Use Your Agent

Now you can interact with your agent using this simple pattern:

In [ ]:
# Complete agent setup in one place
import os
import dotenv
from langchain_openai import AzureChatOpenAI
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# Load environment
dotenv.load_dotenv()

# Create the agent components
memory = MemorySaver()
model = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)
search = TavilySearch(max_results=2)
tools = [search]

# Create the agent with memory
agent = create_react_agent(model, tools, checkpointer=memory)

print("✅ Complete agent created and ready to use!")
print(f"   - Model: {model.model_name}")
print(f"   - Tools: {[tool.name for tool in tools]}")
print(f"   - Memory: Enabled")

## Step 9: Complete Agent Example

Let's put everything together in one clean example that you can use as a template:

### The Complete Pattern

In [ ]:
# New thread (different conversation)
new_config = {"configurable": {"thread_id": "conversation_2"}}

# Ask the same question in a new thread
input_message = {"role": "user", "content": "What's my name and where do I live?"}

print("New conversation (different thread_id):\n")
for step in agent_with_memory.stream({"messages": [input_message]}, new_config, stream_mode="values"):
    last_msg = step["messages"][-1]
    print(f"[{last_msg.type.upper()}] {last_msg.content}\n")

### Test Memory Isolation: Different Thread

Let's prove that different `thread_id` values create separate conversations:

**Amazing!** The agent:
1. **Remembered** Bob lives in San Francisco (from Turn 1)
2. **Used the search tool** to get current weather
3. **Synthesized an answer** using both memory and tool results

This is a **conversational agent** in action!

In [ ]:
# Second message: Ask about the weather where Bob lives
input_message = {"role": "user", "content": "What's the weather where I live?"}

print("Turn 2: Asking about weather (uses memory + search)\n")
for step in agent_with_memory.stream({"messages": [input_message]}, config, stream_mode="values"):
    last_msg = step["messages"][-1]
    print(f"[{last_msg.type.upper()}] {last_msg.content[:200]}...")
    print()

### Test Memory: Follow-Up Question

Now let's ask a follow-up question that requires remembering the previous context:

In [ ]:
# Configuration with thread_id for conversation tracking
config = {"configurable": {"thread_id": "conversation_1"}}

# First message: Introduce yourself
input_message = {"role": "user", "content": "Hi, I'm Bob and I live in San Francisco."}

print("Turn 1: Introducing Bob\n")
for step in agent_with_memory.stream({"messages": [input_message]}, config, stream_mode="values"):
    last_msg = step["messages"][-1]
    print(f"[{last_msg.type.upper()}] {last_msg.content}\n")

### Using Thread IDs for Conversations

Each conversation needs a **thread_id** to track its history:

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create memory checkpointer
memory = MemorySaver()

# Create agent WITH memory
agent_with_memory = create_react_agent(model, tools, checkpointer=memory)

print("Agent with memory created!")

## Step 8: Add Memory for Conversational Agents

So far, our agent is **stateless** - it doesn't remember previous interactions. Let's add **memory** using checkpointing!

### What is Checkpointing?

**Checkpointing** allows agents to:
- ✅ Remember previous messages in a conversation
- ✅ Maintain context across multiple turns
- ✅ Support multi-turn conversations
- ✅ Resume conversations later

### Using `MemorySaver`

LangGraph provides `MemorySaver` for in-memory checkpointing:

In [ ]:
input_message = {"role": "user", "content": "Tell me about Python programming"}

print("Token-by-token streaming:\n")
for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, stream_mode="messages"
):
    # Only show content from the agent node
    if metadata["langgraph_node"] == "agent":
        if hasattr(step, 'content') and step.content:
            print(step.content, end="", flush=True)

print("\n\nStreaming complete!")

### Streaming Tokens with `stream_mode="messages"`

For even finer-grained streaming (token-by-token):

In [ ]:
input_message = {"role": "user", "content": "What is the capital of France?"}

print("Streaming agent responses:\n")
for step in agent_executor.stream({"messages": [input_message]}, stream_mode="values"):
    # Print only the last message in each step
    last_message = step["messages"][-1]
    print(f"[{last_message.type.upper()}] {last_message.content[:100]}...")
    print()

## Step 7: Streaming Agent Responses

For better UX, we can **stream** responses as they're generated instead of waiting for completion.

### Streaming with `stream_mode="values"`

This shows intermediate steps:

**Observation - The ReAct Cycle:**

You should see multiple messages showing the agent's reasoning process:

1. **HUMAN** - Your original question
2. **AI** - Agent decides to use search tool (includes `tool_calls`)
3. **TOOL** - Search results from Tavily
4. **AI** - Agent synthesizes the answer using tool results

This is the **ReAct pattern** in action!

In [ ]:
input_message = {"role": "user", "content": "Search for the weather in San Francisco"}
response = agent_executor.invoke({"messages": [input_message]})

# Print all messages to see the full ReAct cycle
for message in response["messages"]:
    print(f"\n{'='*60}")
    print(f"TYPE: {message.type.upper()}")
    print(f"{'='*60}")
    print(message.content if hasattr(message, 'content') else message)
    if hasattr(message, 'tool_calls') and message.tool_calls:
        print(f"\nTool calls: {message.tool_calls}")

**Observation:**
- The agent responds without using tools
- `response["messages"]` contains the conversation history
- No tool calls were made

### Test 2: Query Requiring Search

Now let's ask something that requires the search tool:

In [ ]:
input_message = {"role": "user", "content": "Hi! My name is Alice."}
response = agent_executor.invoke({"messages": [input_message]})

# Print all messages in the conversation
for message in response["messages"]:
    print(f"{message.type.upper()}: {message.content}\n")

## Step 6: Run the Agent (Stateless)

Let's test our agent with different types of queries.

### Test 1: Simple Query (No Tools Needed)

First, let's ask something that doesn't require tools:

In [ ]:
from langgraph.prebuilt import create_react_agent

# Create a ReAct agent
agent_executor = create_react_agent(model, tools)

print("Agent created successfully!")
print(f"Agent can use {len(tools)} tool(s)")

## Step 5: Create Your First Agent

Now for the exciting part - let's create an **agent** that can actually execute tools!

### Using LangGraph's `create_react_agent`

LangGraph provides a high-level function to create ReAct agents easily:

```python
from langgraph.prebuilt import create_react_agent

agent_executor = create_react_agent(model, tools)
```

**What this does:**
1. Creates an agent with the ReAct pattern (Reason → Act → Observe loop)
2. Automatically handles:
   - Tool invocation
   - Result observation
   - Decision-making (continue or finish)
3. Returns a **stateful graph** that can be executed

Let's create it!

In [ ]:
query = "Search for the weather in San Francisco"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.content}\n")
print(f"Tool calls: {response.tool_calls}")

**Observation:** The LLM responds normally and `tool_calls` is empty `[]` because no tools are needed for a greeting.

### Test 2: Question Requiring Search (Tool Needed)

In [ ]:
query = "Hi! How are you?"
response = model_with_tools.invoke([{"role": "user", "content": query}])

print(f"Message content: {response.content}\n")
print(f"Tool calls: {response.tool_calls}")

## Step 4: Understanding Tool-Calling Behavior

Let's see how the LLM behaves when it knows about tools.

### Test 1: Simple Greeting (No Tools Needed)

In [ ]:
model_with_tools = model.bind_tools(tools)

print("Model now knows about these tools:")
print(f"- {tools[0].name}: {tools[0].description}")

**Notice:** The LLM either makes up an answer or tells you it doesn't have real-time data. It can't actually search for current weather!

### Enable Tool-Calling with `.bind_tools()`

Now let's give the LLM **knowledge of available tools** using `.bind_tools()`:

In [ ]:
query = "What is the weather in San Francisco today?"
response = model.invoke([{"role": "user", "content": query}])

print(f"Response: {response.content}")

### Test Basic LLM (Without Tools)

Let's first see how the LLM responds **without** tools:

In [ ]:
import os
import dotenv
from langchain_openai import AzureChatOpenAI

# Load environment variables
dotenv.load_dotenv()

# Create LLM client for Azure OpenAI
model = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"],
    openai_api_version=os.environ["AZURE_OPENAI_API_VERSION"],
)

print(f"Model initialized: {model.model_name}")

## Step 3: Set Up the LLM with Tool-Calling

Not all LLMs support tool-calling (also called "function calling"). We need to:
1. Choose a model that supports tools
2. Connect to Azure OpenAI (or another provider)
3. Enable tool-calling capabilities

### Connect to Azure OpenAI

Let's use the same Azure OpenAI setup from the RAG notebook:

In [ ]:
# Put tools in a list for the agent
tools = [search]

print(f"Available tools: {[tool.name for tool in tools]}")

### Understanding Tool Results

The search tool returns:
- **query** - What was searched
- **results** - Array of search results with:
  - `title` - Page title
  - `url` - Source URL
  - `content` - Relevant excerpt
  - `score` - Relevance score (0-1)

### Creating a Tool List

Agents can have multiple tools. We'll put our tools in a list:

In [ ]:
from langchain_tavily import TavilySearch

# Create a search tool that returns up to 2 results
search = TavilySearch(max_results=2)

# Test the tool directly (without an agent)
search_results = search.invoke("What is the weather in SF")
print(search_results)

## Step 2: Define and Test Tools

Now let's create our first tool and test it **before** giving it to an agent.

### Creating a Search Tool

We'll use `TavilySearch` - a search engine designed for LLMs that returns structured, relevant results.

### Set Up Tavily API Key

Tavily is a search engine optimized for LLMs. We'll use it as our agent's primary tool.

**Get your free API key:**
1. Go to [Tavily](https://tavily.com/)
2. Sign up for a free account
3. Copy your API key
4. Run the cell below

## Step 1: Understanding Tools

Before building an agent, we need to understand **tools** - the key difference between a regular LLM and an agent.

### What are Tools?

**Tools** are functions or APIs that agents can call to perform actions:

| Tool Type | Examples | Purpose |
|-----------|----------|---------|
| **Search** | Tavily, Google, Bing | Find real-time information |
| **Calculation** | Python REPL, Calculator | Perform computations |
| **Database** | SQL, Vector DB | Query structured data |
| **API Calls** | Weather API, Stock API | Get external data |
| **File Operations** | Read, Write, Delete | Manage files |

### The ReAct Pattern

Agents use the **ReAct** (Reasoning + Acting) pattern:

1. **Reason** - Think about what to do
2. **Act** - Use a tool
3. **Observe** - See the result
4. **Repeat** - Until the task is complete

```
Question: "What's the weather in San Francisco?"

Agent Thinks: "I need current weather data. I'll use the search tool."
Agent Acts: Calls search("weather San Francisco")
Agent Observes: Gets search results with weather info
Agent Thinks: "Now I have the info. I can answer."
Agent Responds: "The weather in SF is 65°F and sunny..."
```

Let's set up our first tool!

In [ ]:
import getpass
import os

# Uncomment to enable LangSmith tracing
# os.environ["LANGSMITH_TRACING"] = "true"
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

### Optional: LangSmith Tracing

LangSmith provides observability for agent applications. It helps you:
- ✅ Debug complex multi-step agent reasoning
- ✅ Monitor tool calls and execution
- ✅ Track performance metrics
- ✅ Visualize agent decision trees

**To enable LangSmith** (optional):
1. Sign up at [LangSmith](https://smith.langchain.com/)
2. Get your API key
3. Run the cell below

**Skip this step** if you don't want tracing for now.

In [ ]:
%pip install -U langgraph langchain-tavily langgraph-checkpoint-sqlite
%pip install -qU "langchain[openai]"